# A/B compartment calculation

Computes the A/B compartment eigenvector (E1) from a cooler matrix using cooltools
`eigs_cis`, then saves the per-bin eigenvector track (`eigenvector_track.tsv` and a
drop-NA `.bed`) for downstream visualization.

Tutorial: https://cooltools.readthedocs.io/en/latest/notebooks/compartments_and_saddles.html


In [ ]:
# Project root — edit for your environment.
PROJ_ROOT = "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj"


In [ ]:
import cooler
import cooltools
# import cooltools.eigdecomp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Following the tutorial here: https://cooltools.readthedocs.io/en/latest/notebooks/compartments_and_saddles.html

In [ ]:
clr = cooler.Cooler(f"{PROJ_ROOT}/output/matrix/403_ear_deep/403_ear_deep_octDeg2.mcool::resolutions/100000")


In [ ]:
import bioframe
bins = clr.bins()[:]
hg38_genome = bioframe.load_fasta(f"{PROJ_ROOT}/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.mito.fasta");
## note the next command may require installing pysam
gc_cov = bioframe.frac_gc(bins[['chrom', 'start', 'end']], hg38_genome)
gc_cov.to_csv('octDeg2_gc_cov_100kb.tsv',index=False,sep='\t')
display(gc_cov)

In [ ]:
view_df = pd.DataFrame({'chrom': clr.chromnames,
                        'start': 0,
                        'end': clr.chromsizes.values,
                        'name': clr.chromnames}
                      )
display(view_df)

In [ ]:
# obtain first 3 eigenvectors
cis_eigs = cooltools.eigs_cis(
                        clr,
                        gc_cov,
                        view_df=view_df,
                        n_eigs=3,
                        )

# cis_eigs[0] returns eigenvalues, here we focus on eigenvectors
eigenvector_track = cis_eigs[1][['chrom','start','end','E1']]

In [ ]:
eigenvector_track

In [ ]:
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable

f, ax = plt.subplots(
    figsize=(15, 10),
)

norm = LogNorm(vmax=0.1)

im = ax.matshow(
    clr.matrix()[:],
    norm=norm,
    cmap='viridis'
);
plt.axis([0,500,500,0])

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.1)
plt.colorbar(im, cax=cax, label='corrected frequencies');
ax.set_ylabel('chr2:0-50Mb')
ax.xaxis.set_visible(False)

ax1 = divider.append_axes("top", size="20%", pad=0.25, sharex=ax)
weights = clr.bins()[:]['weight'].values
ax1.plot([0,500],[0,0],'k',lw=0.25)
ax1.plot( eigenvector_track['E1'].values, label='E1')

ax1.set_ylabel('E1')
ax1.set_xticks([]);


for i in np.where(np.diff( (cis_eigs[1]['E1']>0).astype(int)))[0]:
    ax.plot([0, 500],[i,i],'k',lw=0.5)
    ax.plot([i,i],[0, 500],'k',lw=0.5)


In [ ]:
eigenvector_track.to_csv("./eigenvector_track.tsv",sep="\t")

In [ ]:
df_clean = eigenvector_track.dropna(subset=['E1']).copy()

# Let's check the difference
print(f"Original DataFrame shape: {eigenvector_track.shape}")
print(f"Filtered DataFrame shape: {df_clean.shape}")
print(f"Number of NaN rows removed: {len(eigenvector_track) - len(df_clean)}")

# (Optional but recommended): Reset the index of the new clean DataFrame.
# This ensures the index is neat and continuous (0, 1, 2, 3...).
df_clean.reset_index(drop=True, inplace=True)

# Inspect the cleaned data
print("\nFirst few rows of the cleaned DataFrame:")
print(df_clean.head())

# Save the cleaned DataFrame to a TSV file for use in pycirclize
output_filename = "eigenvector_track_dropNA.bed"
df_clean.to_csv(output_filename, sep='\t', index=False, header=True)

print(f"\nCleaned compartment data saved to: {output_filename}")